# 02 Video Preprocessing & Dataset

Implementation of a memory-efficient `VideoFrameDataset` that performs in-memory frame extraction using OpenCV.

### Key Features:
- **Config-Driven**: All parameters loaded from `video_config.yaml`.
- **In-Memory Extraction**: Frames are extracted on-the-fly, no storage on disk.
- **Uniform Sampling**: Exact `FRAME_COUNT` frames extracted via linear interpolation.
- **Robustness**: Skips corrupted videos and logs warnings.

In [1]:
import cv2
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
import json
import yaml
import logging

# Load Config
with open("../../configs/video_config.yaml", "r") as f:
    config = yaml.safe_load(f)

IMG_SIZE = config["model"]["frame_size"]
FRAME_COUNT = config["model"]["frame_count"]
BATCH_SIZE = config["training"]["batch_size"]
PIN_MEMORY = config["training"]["pin_memory"]
NUM_WORKERS = config["training"]["num_workers"]
DATA_PATH = config["data"]["raw_path"]

# Logging
logging.basicConfig(level=config["logging"]["level"])
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Discovery Utilities

In [2]:
def discover_dataset_parts(base_path):
    base_path = Path(base_path)
    if not base_path.exists(): return []
    return sorted([
        base_path / d for d in os.listdir(base_path)
        if d.startswith("dfdc_train_part_") and (base_path / d).is_dir()
    ], key=lambda x: int(x.name.split('_')[-1]))

def get_video_list(parts_paths):
    video_list = []
    for part_path in parts_paths:
        metadata_path = part_path / "metadata.json"
        if not metadata_path.exists(): continue
        with open(metadata_path, "r") as f:
            metadata = json.load(f)
        for filename, info in metadata.items():
            video_path = part_path / filename
            if video_path.exists():
                label = 1 if info["label"] == "FAKE" else 0
                video_list.append((str(video_path), label))
    return video_list

## 2. VideoFrameDataset Class

In [3]:
class VideoFrameDataset(Dataset):
    def __init__(self, video_list, num_frames=32, transform=None):
        self.video_list = video_list
        self.num_frames = num_frames
        self.transform = transform
        
    def __len__(self):
        return len(self.video_list)
    
    def _get_frames(self, video_path):
        try:
            cap = cv2.VideoCapture(video_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            if total_frames <= 0:
                cap.release()
                return None
            
            # Uniform sampling indices
            indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
            
            frames = []
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if not ret:
                    # Fill with black frame if read fails
                    frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
                    continue
                
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                
                if self.transform:
                    frame = self.transform(frame)
                
                frames.append(frame)
                
            cap.release()
            return torch.stack(frames)
        except Exception as e:
            logger.warning(f"Error processing {video_path}: {e}")
            return None
    
    def __getitem__(self, index):
        video_path, label = self.video_list[index]
        frames = self._get_frames(video_path)
        
        if frames is None:
            # Return zero tensor to avoid crashing batch
            frames = torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
            
        return frames, torch.tensor(label, dtype=torch.float32)

## 3. DataLoader Setup & Test

In [4]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

parts = discover_dataset_parts(DATA_PATH)
all_videos = get_video_list(parts)

if all_videos:
    dataset = VideoFrameDataset(all_videos[:10], num_frames=FRAME_COUNT, transform=transform)
    loader = DataLoader(dataset, batch_size=2, shuffle=True, pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS)
    
    frames, labels = next(iter(loader))
    print(f"Batch frames shape: {frames.shape}") # Expected: (Batch, T, C, H, W)
    print(f"Batch labels: {labels}")
else:
    print("No videos found.")